<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/systems/conv-bn-fusion/conv-bn-fusion.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("conv-bn-fusion", ...)`


# Inference-Time Re-parameterization: Conv–BN Fusion and RepVGG Merging

**Difficulty**: 🟢 Easy

### Problem Statement

At inference time BatchNorm no longer normalizes anything: with frozen running statistics it is a fixed
per-channel affine map. Deployment toolchains therefore **fold** it into the convolution (or linear layer)
in front of it. That saves one memory-bound pass over the activation tensor and one kernel launch per
block, and it is a prerequisite for int8 deployment: the folded weights are what get quantized (see
`v4/systems/int8-quantization`). PyTorch ships the fold as `torch.nn.utils.fusion.fuse_conv_bn_eval`,
which `torch.ao.quantization.fuse_modules` calls for eval-mode models. TensorRT and ONNX Runtime apply it automatically when
they optimize a graph. `v4/dl-fundamentals/normalization-layers` derives BatchNorm's train and eval
behaviour and includes a three-line folding demo. Here you write the fold in full generality: grouped and
depthwise convs, no conv bias, `affine=False`, any `eps`, and Linear + BatchNorm1d.

With running mean $\mu_c$, running variance $\sigma_c^2$ and affine parameters $\gamma_c, \beta_c$,
eval-mode BN applied to the conv output $z = W * x + b$ is

$$\mathrm{BN}(z)_c = \gamma_c\,\frac{z_c - \mu_c}{\sqrt{\sigma_c^2 + \epsilon}} + \beta_c = s_c\, z_c + \beta_c - s_c\mu_c,
\qquad s_c = \frac{\gamma_c}{\sqrt{\sigma_c^2 + \epsilon}}.$$

Scaling output channel $c$ is the same as scaling that channel's filter, so

$$W'_{c} = s_c\, W_{c}, \qquad b'_c = s_c\,(b_c - \mu_c) + \beta_c, \qquad \mathrm{BN}(W * x + b) = W' * x + b'.$$

The same algebra drives **structural re-parameterization**. RepVGG (Ding et al., *RepVGG: Making
VGG-style ConvNets Great Again*, CVPR 2021) trains each block with three parallel branches: a 3×3
conv + BN, a 1×1 conv + BN, and an identity branch that is a BN alone. This trains like a ResNet. After
training the three branches collapse (§3.3, Eqs. 1–3) into **one 3×3 conv**, so the deployed network is a
plain stack of 3×3 convs and ReLUs:

$$\mathrm{BN}_3(W_3 * x) + \mathrm{BN}_1(W_1 * x) + \mathrm{BN}_0(x) \;=\; \big(W_3' + \mathrm{pad}(W_1') + K_{\mathrm{id}}'\big) * x + (b_3' + b_1' + b_0').$$

Each branch is folded with its own BN. The 1×1 kernel is zero-padded to 3×3 around the **centre** tap.
The identity is written as a conv whose kernel copies input channel $c$ to output channel $c$, and then
its BN is folded in.

---

### Requirements

1. **`fuse_conv_bn(conv, bn) -> conv module`**
   - `conv` is an `nn.Conv1d` / `nn.Conv2d` / `nn.Conv3d` (not transposed), with any `stride`,
     `padding`, `dilation`, `groups` (including depthwise) and `padding_mode`, with or without a bias.
     `bn` is the matching `BatchNorm1d/2d/3d` with `num_features == conv.out_channels`,
     `track_running_stats=True`, `affine` True **or False**, and any `eps`. Both are in eval mode.
   - Return a **new** module of the same class as `conv` whose forward equals `bn(conv(x))` with BN in
     eval mode, i.e. using the **running** statistics: $W' = s \cdot W$ along the **output-channel** axis
     (dim 0 of the weight, also for grouped convs) and $b' = s\,(b - \mu) + \beta$. Use $b = 0$ if
     `conv.bias is None`, $\gamma = 1, \beta = 0$ if `bn.affine` is False, and $\epsilon$ = `bn.eps`.
   - The result keeps all of `conv`'s hyperparameters and dtype. It **always** has a bias, and its
     `weight` and `bias` are `nn.Parameter`s.
   - `conv` and `bn` must be left untouched (values, parameters, train/eval mode), and the result must not
     share tensors with `conv`. This is the contract of `torch.nn.utils.fusion.fuse_conv_bn_eval`.
2. **`fuse_linear_bn(linear, bn) -> nn.Linear`**: the same for `nn.Linear(in, out)` followed by
   `nn.BatchNorm1d(out)` on `(N, out)` activations. Handle `bias=False` and `affine=False`.
3. **`repvgg_merge(branch_3x3, branch_1x1, identity_bn=None) -> nn.Conv2d`**
   - `branch_3x3 = (conv3, bn3)` with `conv3 = nn.Conv2d(C_in, C_out, 3, stride, padding=1, groups=g)`.
     `branch_1x1 = (conv1, bn1)` with `conv1 = nn.Conv2d(C_in, C_out, 1, stride, padding=0, groups=g)`.
     Both use the same `stride` ∈ {1, 2} and the same `groups` (1, a divisor, or depthwise), with
     dilation 1 and zero padding. The convs may or may not have a bias. `bn3`, `bn1` are
     `BatchNorm2d(C_out)`.
   - `identity_bn` is a `BatchNorm2d(C_in)` or `None`. An identity branch only exists when
     `C_in == C_out` and `stride == 1`. If `identity_bn` is given in any other configuration, **raise
     `ValueError`**.
   - Return `nn.Conv2d(C_in, C_out, 3, stride, padding=1, groups=g, bias=True)` whose output equals
     `bn3(conv3(x)) + bn1(conv1(x)) [+ identity_bn(x)]` in eval mode. Leave the inputs untouched.

---

### Constraints

- ✅ `copy.deepcopy`, `F.pad`, reshaping/broadcasting, `torch.no_grad()`.
- ✅ Use `torch.nn.utils.fusion.fuse_conv_bn_eval` in the test cells to check yourself.
- ❌ Don't call `torch.nn.utils.fusion.*` or `torch.ao.quantization.fuse_modules` inside your functions.
  The grader detects it.
- ❌ Don't fit the fused weights from sample inputs and outputs. Derive them.
- ❌ Don't modify the input modules (no `conv.weight.data *= ...`, no `copy.copy`, which shares the
  parameter dict).

---

<details>
  <summary>💡 Hint</summary>

  - What is the shape of a `Conv2d` weight when `groups=g`? Which axis indexes the output channels?
  - What does BN do to its input when `affine=False`? What should $b$ be when the conv has no bias?
  - In a 3×3 conv with padding 1, which kernel tap reads the same input pixel as a 1×1 conv with
    padding 0 and the same stride?
  - How would you write "copy input channel $c$ to output channel $c$" as a conv kernel? In a grouped
    conv, output channel $o$ only sees the $C/g$ input channels of its group. Which *local* index does
    channel $o$ have there?

</details>

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.fusion import fuse_conv_bn_eval     # for checking your answer only

In [ ]:
torch.manual_seed(0)


def randomize_bn(bn, seed):
    # Give a BatchNorm the kind of running statistics / affine params it has after training.
    g = torch.Generator().manual_seed(seed)
    C = bn.num_features
    with torch.no_grad():
        bn.running_mean.copy_(torch.randn(C, generator=g) * 2.0)
        bn.running_var.copy_(torch.exp(torch.randn(C, generator=g)))
        if bn.affine:
            bn.weight.copy_(torch.randn(C, generator=g) * 0.8 + 0.5)
            bn.bias.copy_(torch.randn(C, generator=g))
    return bn.eval()


class RepVGGBlock(nn.Module):
    # Training-time RepVGG block (Ding et al., 2021). The ReLU is applied outside the block.
    def __init__(self, cin, cout, stride=1, groups=1):
        super().__init__()
        self.conv3 = nn.Conv2d(cin, cout, 3, stride, padding=1, groups=groups, bias=False)
        self.bn3 = nn.BatchNorm2d(cout)
        self.conv1 = nn.Conv2d(cin, cout, 1, stride, padding=0, groups=groups, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.bn_id = nn.BatchNorm2d(cin) if cin == cout and stride == 1 else None

    def forward(self, x):
        y = self.bn3(self.conv3(x)) + self.bn1(self.conv1(x))
        if self.bn_id is not None:
            y = y + self.bn_id(x)
        return y


# A small RepVGG-style network, trained for a few steps so the BNs hold real running statistics.
net = nn.Sequential(RepVGGBlock(3, 8, stride=2), nn.ReLU(),
                    RepVGGBlock(8, 8), nn.ReLU(),
                    RepVGGBlock(8, 8, groups=2), nn.ReLU())
teacher = nn.Sequential(nn.Conv2d(3, 8, 3, 2, 1), nn.Tanh(), nn.Conv2d(8, 8, 3, 1, 1))
opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9)
for step in range(40):
    xb = torch.randn(16, 3, 16, 16) * 1.5 + 0.3
    with torch.no_grad():
        target = torch.relu(teacher(xb))
    loss = F.mse_loss(net(xb), target)
    opt.zero_grad()
    loss.backward()
    opt.step()
net.eval()
print(f"trained for 40 steps, final loss {loss.item():.4f}")
print("block 2 bn3.running_var[:4] =", [round(v, 3) for v in net[2].bn3.running_var[:4].tolist()])

In [ ]:
def fuse_conv_bn(conv, bn):
    """Fold an eval-mode BatchNorm into the conv that precedes it.

    Args:
        conv: nn.Conv1d/2d/3d, weight (C_out, C_in // groups, *kernel), bias (C_out,) or None.
        bn: matching BatchNorm with C_out features (running stats; affine may be False).

    Returns:
        A NEW module of type(conv), same hyperparameters, with weight W' and bias b' (always present)
        such that new(x) == bn(conv(x)) in eval mode. conv and bn are not modified.
    """
    # TODO: s = gamma / sqrt(running_var + eps)  (gamma = 1, beta = 0 when bn.affine is False)
    # TODO: W' = W * s along the output-channel axis; b' = (b - running_mean) * s + beta  (b = 0 if no bias)
    # TODO: put W', b' into a fresh copy of conv as nn.Parameters
    raise NotImplementedError

In [ ]:
def fuse_linear_bn(linear, bn):
    """Fold an eval-mode BatchNorm1d(out_features) into the nn.Linear before it.

    Args:
        linear: nn.Linear(in_features, out_features), bias optional.
        bn: nn.BatchNorm1d(out_features), affine may be False.

    Returns:
        A NEW nn.Linear with new(x) == bn(linear(x)) for x of shape (N, in_features), eval mode.
    """
    # TODO: same algebra as fuse_conv_bn; the output axis of a Linear weight (out, in) is dim 0
    raise NotImplementedError

In [ ]:
def repvgg_merge(branch_3x3, branch_1x1, identity_bn=None):
    """Merge a RepVGG block (3x3 conv+BN, 1x1 conv+BN, optional identity BN) into one 3x3 conv.

    Args:
        branch_3x3: (conv3, bn3), conv3 = Conv2d(C_in, C_out, 3, stride, padding=1, groups=g).
        branch_1x1: (conv1, bn1), conv1 = Conv2d(C_in, C_out, 1, stride, padding=0, groups=g).
        identity_bn: BatchNorm2d(C_in) or None. Only valid if C_in == C_out and stride == 1,
            otherwise raise ValueError.

    Returns:
        nn.Conv2d(C_in, C_out, 3, stride, padding=1, groups=g, bias=True) equal to
        bn3(conv3(x)) + bn1(conv1(x)) [+ identity_bn(x)] in eval mode.
    """
    # TODO: fuse each conv branch with its own BN
    # TODO: zero-pad the fused 1x1 kernel to 3x3 so it sits at the centre tap
    # TODO: identity branch -> a 3x3 kernel that copies input channel o to output channel o (mind groups), then fold identity_bn into it
    # TODO: sum kernels and biases into a new Conv2d (validate the identity configuration first)
    raise NotImplementedError

In [ ]:
# Test 1: fuse_conv_bn == eval-mode bn(conv(x)), and agrees with PyTorch's own fusion
configs = [
    dict(cin=4, cout=6, k=3, padding=1),                          # plain conv with bias
    dict(cin=3, cout=5, k=3, stride=2, padding=1, bias=False),    # no conv bias
    dict(cin=6, cout=6, k=3, padding=1, groups=6),                # depthwise
    dict(cin=4, cout=8, k=1, groups=2),                           # grouped 1x1
]
for i, c in enumerate(configs):
    conv = nn.Conv2d(c["cin"], c["cout"], c["k"], stride=c.get("stride", 1), padding=c.get("padding", 0),
                     groups=c.get("groups", 1), bias=c.get("bias", True)).double().eval()
    bn = randomize_bn(nn.BatchNorm2d(c["cout"]).double(), seed=i)
    x = torch.randn(2, c["cin"], 9, 9, dtype=torch.float64)
    fused = fuse_conv_bn(conv, bn)
    assert isinstance(fused, nn.Conv2d) and fused is not conv, "return a NEW Conv2d"
    assert (fused.stride, fused.padding, fused.groups) == (conv.stride, conv.padding, conv.groups)
    assert isinstance(fused.bias, nn.Parameter), "the fused conv always needs a bias"
    with torch.no_grad():
        ref = bn(conv(x))
        out = fused(x)
        torch_out = fuse_conv_bn_eval(conv, bn)(x)
    assert torch.allclose(out, ref, rtol=1e-10, atol=1e-10), f"config {c}: max err {(out - ref).abs().max():.2e}"
    assert torch.allclose(out, torch_out, rtol=1e-10, atol=1e-10)
print("Test 1 passed: grouped / depthwise / bias-free convs fold exactly")

In [ ]:
# Test 2: affine=False, a non-default eps, Conv1d, and the inputs stay untouched
conv = nn.Conv2d(4, 6, 3, padding=1, bias=False).double().eval()
bn = randomize_bn(nn.BatchNorm2d(6, affine=False, eps=1e-3).double(), seed=10)
bn.running_var.mul_(1e-3)                       # variances comparable to eps: eps placement matters
w_before, var_before = conv.weight.detach().clone(), bn.running_var.clone()
fused = fuse_conv_bn(conv, bn)
assert conv.bias is None, "the original conv must keep bias=None"
assert torch.equal(conv.weight, w_before) and torch.equal(bn.running_var, var_before), \
    "fuse_conv_bn modified its inputs"
assert fused.weight.data_ptr() != conv.weight.data_ptr(), "the fused conv shares its weight with the input"
x = torch.randn(2, 4, 8, 8, dtype=torch.float64)
with torch.no_grad():
    assert torch.allclose(fused(x), bn(conv(x)), rtol=1e-10, atol=1e-10)

conv1d = nn.Conv1d(3, 4, 5, padding=2).double().eval()
bn1d = randomize_bn(nn.BatchNorm1d(4).double(), seed=11)
fused1d = fuse_conv_bn(conv1d, bn1d)
assert isinstance(fused1d, nn.Conv1d)
x1 = torch.randn(2, 3, 20, dtype=torch.float64)
with torch.no_grad():
    assert torch.allclose(fused1d(x1), bn1d(conv1d(x1)), rtol=1e-10, atol=1e-10)
print("Test 2 passed")

In [ ]:
# Test 3: Linear + BatchNorm1d
for i, (lin, bn) in enumerate([(nn.Linear(7, 5), nn.BatchNorm1d(5)),
                               (nn.Linear(6, 4, bias=False), nn.BatchNorm1d(4, affine=False))]):
    lin, bn = lin.double().eval(), randomize_bn(bn.double(), seed=20 + i)
    fused = fuse_linear_bn(lin, bn)
    assert isinstance(fused, nn.Linear) and fused is not lin
    x = torch.randn(10, lin.in_features, dtype=torch.float64)
    with torch.no_grad():
        assert torch.allclose(fused(x), bn(lin(x)), rtol=1e-10, atol=1e-10)
print("Test 3 passed")

In [ ]:
# Test 4: RepVGG — merge every trained block; the plain conv/ReLU stack must equal the trained net
plain_layers = []
for m in net:
    if isinstance(m, RepVGGBlock):
        plain_layers.append(repvgg_merge((m.conv3, m.bn3), (m.conv1, m.bn1), m.bn_id))
    else:
        plain_layers.append(m)
plain = nn.Sequential(*plain_layers).eval()
for layer, block in zip(plain_layers[::2], net[::2]):
    assert isinstance(layer, nn.Conv2d) and layer.kernel_size == (3, 3) and layer.padding == (1, 1)
    assert layer.stride == block.conv3.stride and layer.groups == block.conv3.groups
x = torch.randn(4, 3, 16, 16)
with torch.no_grad():
    y_train_time, y_deploy = net(x), plain(x)
err = (y_train_time - y_deploy).abs().max().item()
print(f"trained 3-branch net vs merged plain net: max |diff| = {err:.2e} (float32)")
assert err < 1e-4

# float64: identity branch with groups, depthwise, and an impossible identity
for cin, cout, stride, groups in [(6, 6, 1, 1), (8, 8, 1, 2), (6, 6, 1, 6), (4, 8, 2, 4)]:
    blk = RepVGGBlock(cin, cout, stride, groups).double().eval()
    for j, bn in enumerate((blk.bn3, blk.bn1, blk.bn_id)):
        if bn is not None:
            randomize_bn(bn, seed=100 * j + cin + cout + groups)   # different statistics per branch
    merged = repvgg_merge((blk.conv3, blk.bn3), (blk.conv1, blk.bn1), blk.bn_id)
    xd = torch.randn(2, cin, 7, 9, dtype=torch.float64)
    with torch.no_grad():
        assert torch.allclose(merged(xd), blk(xd), rtol=1e-10, atol=1e-10), (cin, cout, stride, groups)
blk = RepVGGBlock(4, 8).eval()
try:
    repvgg_merge((blk.conv3, blk.bn3), (blk.conv1, blk.bn1), nn.BatchNorm2d(4).eval())
    raise AssertionError("an identity branch with in_channels != out_channels must raise ValueError")
except ValueError:
    pass
print("Test 4 passed: the RepVGG net collapses into a plain stack of 3x3 convs")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("conv-bn-fusion", fuse_conv_bn=fuse_conv_bn, fuse_linear_bn=fuse_linear_bn, repvgg_merge=repvgg_merge)